In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/anaghabpoojari11/ensemble/fusion_embeddings.npz
/kaggle/input/notebooks/anaghabpoojari11/ensemble/best_fusion_head.pth
/kaggle/input/notebooks/anaghabpoojari11/ensemble/__results__.html
/kaggle/input/notebooks/anaghabpoojari11/ensemble/fusion_confusion_matrix.png
/kaggle/input/notebooks/anaghabpoojari11/ensemble/__notebook__.ipynb
/kaggle/input/notebooks/anaghabpoojari11/ensemble/fusion_classification_report.csv
/kaggle/input/notebooks/anaghabpoojari11/ensemble/__output__.json
/kaggle/input/notebooks/anaghabpoojari11/ensemble/custom.css
/kaggle/input/notebooks/anaghabpoojari11/ensemble/__results___files/__results___9_1.png
/kaggle/input/notebooks/anaghabpoojari11/ensemble/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/anaghabpoojari11/surgfusion/classification_report.csv
/kaggle/input/notebooks/anaghabpoojari11/surgfusion/training_curves.png
/kaggle/input/notebooks/anaghabpoojari11/surgfusion/confusion_matrix.png
/kaggle/input/notebooks/ana

In [4]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.2 MB/s eta 0:00:00


In [2]:
CLIP_A = '/kaggle/input/datasets/anaghabpoojari11/videoss/Suturing_D001_capture2.mp4'
CLIP_B = '/kaggle/input/datasets/anaghabpoojari11/videoss/Needle_Passing_C003_capture1.mp4'
GESTURE_WEIGHTS = '/kaggle/input/notebooks/anaghabpoojari11/surgfusion/best_c2plus1d_gesture.pth'
PHASE_WEIGHTS   = '/kaggle/input/notebooks/argrand/notebook7478de4a3c/checkpoints/r3d18_ep50.pth'
FUSION_WEIGHTS  = '/kaggle/input/notebooks/anaghabpoojari11/ensemble/best_fusion_head.pth'
YOLO_WEIGHTS    = '/kaggle/input/models/anaghabpoojari11/object/pytorch/default/1/best.pt'

In [5]:
import torch, torch.nn as nn, torch.nn.functional as F
import cv2, numpy as np
from torchvision.models.video import r2plus1d_18, r3d_18
from ultralytics import YOLO
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
GESTURE_CLASSES = ['G1','G11','G12','G13','G14','G15','G2','G3','G4','G5','G6','G8','G9']  # confirm this matches your saved GESTURE_TO_IDX order
PHASE_CLASSES = sorted([
    'AutoLaparo_dividing_ligament_and_peritoneum','AutoLaparo_dividing_uterine_vessels_and_ligament',
    'AutoLaparo_preparation','AutoLaparo_specimen_removal','AutoLaparo_suturing',
    'AutoLaparo_transecting_the_vagina','AutoLaparo_washing',
    'CholecT50_carlot_triangle_dissection','CholecT50_cleaning_and_coagulation',
    'CholecT50_gallbladder_dissection','CholecT50_gallbladder_extraction','CholecT50_preparation',
    'endovis2019_calot_triangle_dissection','endovis2019_cleaning_and_coagulation',
    'endovis2019_clipping_and_cutting','endovis2019_galbladder_dissection',
    'endovis2019_galbladder_packaging','endovis2019_galbladder_retraction','endovis2019_preparation'
])  # confirm exact order matches your teammate's `folders` variable
 
gesture_model = r2plus1d_18(weights=None)
gesture_model.fc = nn.Linear(gesture_model.fc.in_features, 13)
gesture_model.load_state_dict(torch.load(GESTURE_WEIGHTS, map_location=device))
gesture_model = gesture_model.to(device).eval()
 
phase_model = r3d_18(weights=None)
phase_model.fc = nn.Linear(phase_model.fc.in_features, 19)
phase_ckpt = torch.load(PHASE_WEIGHTS, map_location=device)
phase_model.load_state_dict(phase_ckpt['model_state_dict'])
phase_model = phase_model.to(device).eval()
 
class WeightedFusionHead(nn.Module):
    def __init__(self, emb_dim=512, num_classes=13, proj_dim=256):
        super().__init__()
        self.proj_gesture = nn.Linear(emb_dim, proj_dim)
        self.proj_phase = nn.Linear(emb_dim, proj_dim)
        self.raw_weights = nn.Parameter(torch.tensor([1.0, 0.0]))
        self.classifier = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(proj_dim, num_classes))
    def forward(self, g_emb, p_emb):
        w = F.softmax(self.raw_weights, dim=0)
        fused = w[0]*self.proj_gesture(g_emb) + w[1]*self.proj_phase(p_emb)
        return self.classifier(fused), w
 
fusion_model = WeightedFusionHead().to(device)
fusion_model.load_state_dict(torch.load(FUSION_WEIGHTS, map_location=device))
fusion_model.eval()
 
yolo_model = YOLO(YOLO_WEIGHTS)
 
print("All models loaded.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All models loaded.


In [6]:
def load_clip_as_tensor(video_path, clip_len=16, resize=(112,112)):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(total-1,0), clip_len).astype(int)
    frames = []
    for fi in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((resize[0], resize[1], 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, resize)
        frames.append(frame)
    cap.release()
    clip = np.stack(frames).astype(np.float32) / 255.0
    return torch.from_numpy(clip).permute(3,0,1,2).unsqueeze(0)
 
clip_t = load_clip_as_tensor(CLIP_A).to(device)
mean = torch.tensor([0.43216,0.394666,0.37645]).view(1,3,1,1,1).to(device)
std  = torch.tensor([0.22803,0.22145,0.216989]).view(1,3,1,1,1).to(device)
gesture_input = (clip_t - mean) / std
phase_input = clip_t
 
with torch.no_grad():
    g_out = gesture_model(gesture_input)
    g_pred = GESTURE_CLASSES[g_out.argmax(1).item()]
    g_conf = F.softmax(g_out, dim=1).max().item()
 
    p_out = phase_model(phase_input)
    p_pred = PHASE_CLASSES[p_out.argmax(1).item()]
    p_conf = F.softmax(p_out, dim=1).max().item()
 
    g_backbone = nn.Sequential(*list(gesture_model.children())[:-1])
    p_backbone = nn.Sequential(*list(phase_model.children())[:-1])
    g_emb = g_backbone(gesture_input).flatten(1)
    p_emb = p_backbone(phase_input).flatten(1)
    fused_out, weights = fusion_model(g_emb, p_emb)
    fused_pred = GESTURE_CLASSES[fused_out.argmax(1).item()]
    fused_conf = F.softmax(fused_out, dim=1).max().item()
 
print(f"Gesture-only: {g_pred} ({g_conf:.2f})")
print(f"Phase (cross-domain, out-of-distribution): {p_pred} ({p_conf:.2f})")
print(f"Fused prediction: {fused_pred} ({fused_conf:.2f})  |  weights g={weights[0].item():.2f} p={weights[1].item():.2f}")

Gesture-only: G6 (0.46)
Phase (cross-domain, out-of-distribution): CholecT50_gallbladder_extraction (0.25)
Fused prediction: G6 (0.51)  |  weights g=0.74 p=0.26


In [7]:
FONT = cv2.FONT_HERSHEY_SIMPLEX
 
def make_title_card(text_lines, size=(1280,720), duration_sec=2, fps=25, bg_color=(30,30,30)):
    frames = []
    frame = np.full((size[1], size[0], 3), bg_color, dtype=np.uint8)
    y = size[1]//2 - (len(text_lines)*40)//2
    for line in text_lines:
        (tw, th), _ = cv2.getTextSize(line, FONT, 1.2, 2)
        x = (size[0]-tw)//2
        cv2.putText(frame, line, (x, y), FONT, 1.2, (255,255,255), 2, cv2.LINE_AA)
        y += 50
    for _ in range(duration_sec*fps):
        frames.append(frame.copy())
    return frames
 
def overlay_text_block(frame, lines, origin=(20,30), color=(0,255,255)):
    y = origin[1]
    for line in lines:
        cv2.putText(frame, line, (origin[0], y), FONT, 0.6, (0,0,0), 3, cv2.LINE_AA)   # outline
        cv2.putText(frame, line, (origin[0], y), FONT, 0.6, color, 1, cv2.LINE_AA)      # text
        y += 26
    return frame
 
def process_clip_a_frames(video_path, size=(1280,720)):
    cap = cv2.VideoCapture(video_path)
    out_frames = []
    overlay_lines = [
        "GESTURE + PHASE + FUSION",
        f"Gesture-only:  {g_pred}  ({g_conf:.2f})",
        f"Phase (cross-domain): {p_pred[:35]}  ({p_conf:.2f})",
        f"FUSED PREDICTION: {fused_pred}  ({fused_conf:.2f})",
        f"Fusion weights -> gesture {weights[0].item():.2f} / phase {weights[1].item():.2f}",
    ]
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, size)
        frame = overlay_text_block(frame, overlay_lines)
        out_frames.append(frame)
    cap.release()
    return out_frames
 
def process_clip_b_frames(video_path, size=(1280,720), sample_every=2):
    cap = cv2.VideoCapture(video_path)
    out_frames = []
    idx = 0
    last_annotated = None
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, size)
        if idx % sample_every == 0 or last_annotated is None:
            results = yolo_model(frame, verbose=False)[0]
            annotated = results.plot()
            last_annotated = annotated
        else:
            annotated = last_annotated
        annotated = overlay_text_block(annotated, ["OBJECT DETECTION (independent module)"], color=(0,140,255))
        out_frames.append(annotated)
        idx += 1
    cap.release()
    return out_frames

In [10]:
from pathlib import Path
FPS = 25
SIZE = (1280, 720)
OUTPUT_PATH = "/kaggle/working/surgical_demo.mp4"
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
title1 = make_title_card(
    [
        "Multimodal Surgical Video Understanding",
        "Input: JIGSAWS Gesture Clip",
        "-> Gesture + Phase + Weighted Fusion"
    ],
    size=SIZE,
    fps=FPS
)
clip_a_frames = process_clip_a_frames(
    CLIP_A,
    size=SIZE
)
title2 = make_title_card(
    [
        "Input: Laparoscopic Clip",
        "-> Object / Tool Detection (independent module)"
    ],
    size=SIZE,
    fps=FPS
)
clip_b_frames = process_clip_b_frames(
    CLIP_B,
    size=SIZE
)
summary = make_title_card(
    [
        "Summary",
        "Gesture-only acc: 90.2%  |  Fusion acc: 91.1%",
        "Fusion weights: gesture 0.74 / phase 0.26",
        "Object detection: reported independently"
    ],
    size=SIZE,
    duration_sec=3,
    fps=FPS
)
all_frames = (
    title1
    + clip_a_frames
    + title2
    + clip_b_frames
    + summary
)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    FPS,
    SIZE
)

if not writer.isOpened():
    raise RuntimeError(
        f"Could not open VideoWriter for: {OUTPUT_PATH}"
    )

for frame in all_frames:
    if frame.shape[1] != SIZE[0] or frame.shape[0] != SIZE[1]:
        frame = cv2.resize(frame, SIZE)

    writer.write(frame)

writer.release()

print(f"Done. {len(all_frames)} frames written to:")
print(OUTPUT_PATH)

Done. 6567 frames written to:
/kaggle/working/surgical_demo.mp4
